<div style="direction: rtl; white-space: normal; line-height: 1;">
در این بخش، مسیر فایل‌ها، مدل embedding، دیتاست قرآن، بردارها و کلاینت Qwen داخل یک کلاس واحد بارگذاری می‌شوند تا اجرای کامل RAG از یک نقطه کنترل شود.
</div>

In [1]:
# Build the reusable Quran RAG pipeline class

from pathlib import Path
import os
import json
import hashlib

import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
from sentence_transformers import SentenceTransformer


class QuranRAGPipeline:
    def __init__(
        self,
        project_root=None,
        embedding_model_name="intfloat/multilingual-e5-small",
        top_k=3,
        use_cache=True
    ):
        self.project_root = (
            Path(project_root)
            if project_root
            else (
                Path.cwd().parent
                if Path.cwd().name == "notebooks"
                else Path.cwd()
            )
        )

        load_dotenv(self.project_root / ".env")

        self.dataset_file = (
            self.project_root
            / "data"
            / "processed"
            / "quran_dataset_clean.json"
        )

        self.embeddings_file = (
            self.project_root
            / "data"
            / "embeddings"
            / "quran_embeddings.npy"
        )

        self.cache_dir = self.project_root / "data" / "cache"
        self.cache_dir.mkdir(parents=True, exist_ok=True)

        self.cache_file = self.cache_dir / "rag_cache.json"

        self.embedding_model_name = embedding_model_name
        self.top_k = top_k
        self.use_cache = use_cache

        self.api_key = (
            os.getenv("ARVAN_CHAT_API_KEY")
            or os.getenv("ARVAN_AI_API_KEY")
        )

        self.chat_url = os.getenv("ARVAN_CHAT_URL")
        self.chat_model = (
            os.getenv("ARVAN_CHAT_MODEL")
            or "Qwen3-30B-A3B"
        )

        if not self.api_key:
            raise ValueError("Arvan API key is missing.")

        if not self.chat_url:
            raise ValueError("ARVAN_CHAT_URL is missing.")

        with open(self.dataset_file, "r", encoding="utf-8") as file:
            self.quran_data = json.load(file)

        self.quran_embeddings = np.load(self.embeddings_file)

        self.embedding_model = SentenceTransformer(
            self.embedding_model_name,
            device="cpu"
        )

        self.client = OpenAI(
            api_key=self.api_key,
            base_url=self.chat_url
        )

        self.cache = self._load_cache()

    def _load_cache(self):
        if not self.use_cache or not self.cache_file.exists():
            return {}

        try:
            with open(self.cache_file, "r", encoding="utf-8") as file:
                return json.load(file)
        except Exception:
            return {}

    def _save_cache(self):
        if not self.use_cache:
            return

        with open(self.cache_file, "w", encoding="utf-8") as file:
            json.dump(
                self.cache,
                file,
                ensure_ascii=False,
                indent=2
            )

    def _make_cache_key(self, question):
        normalized_question = question.strip().lower()

        raw_key = (
            f"{self.embedding_model_name}|"
            f"{self.chat_model}|"
            f"{self.top_k}|"
            f"{normalized_question}"
        )

        return hashlib.sha256(
            raw_key.encode("utf-8")
        ).hexdigest()


print("QuranRAGPipeline base class is ready.")

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


QuranRAGPipeline base class is ready.


<div style="direction: rtl; white-space: normal; line-height: 1;">
در این بخش، کلاس سؤال را به embedding تبدیل می‌کند، آیات مرتبط را بازیابی می‌کند و آن‌ها را به متن مرجع ساختاریافته تبدیل می‌کند.
</div>

In [2]:
# Add retrieval and context methods to the pipeline

def retrieve(self, question, top_k=None):
    """
    Retrieve the most relevant Quran verses
    """
    if not isinstance(question, str) or not question.strip():
        raise ValueError("Question must be a non-empty string.")

    selected_top_k = top_k or self.top_k
    selected_top_k = min(selected_top_k, len(self.quran_data))

    query_embedding = self.embedding_model.encode(
        [f"query: {question.strip()}"],
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False
    )[0]

    scores = self.quran_embeddings @ query_embedding
    top_indices = np.argsort(scores)[::-1][:selected_top_k]

    results = []

    for rank, index in enumerate(top_indices, start=1):
        item = self.quran_data[index]

        results.append({
            "rank": rank,
            "score": float(scores[index]),
            "surah": item["surah"],
            "ayah": item["ayah"],
            "arabic": item["arabic"],
            "fooladvand": item["fooladvand"],
            "ansarian": item["ansarian"],
        })

    return results


def build_context(self, results):
    """
    Convert retrieved verses into structured context
    """
    context_parts = []

    for result in results:
        context_parts.append(
            f"[سوره {result['surah']}، آیه {result['ayah']}]\n"
            f"متن عربی: {result['arabic']}\n"
            f"ترجمه فولادوند: {result['fooladvand']}\n"
            f"ترجمه انصاریان: {result['ansarian']}"
        )

    return "\n\n".join(context_parts)


QuranRAGPipeline.retrieve = retrieve
QuranRAGPipeline.build_context = build_context

print("Retrieval and context methods are ready.")

Retrieval and context methods are ready.


<div style="direction: rtl; white-space: normal; line-height: 1;">
در این بخش، مدل فقط رتبه آیه‌هایی را انتخاب می‌کند که مستقیم به سؤال پاسخ می‌دهند و اجازه تولید متن آزاد ندارد
</div>

In [16]:
# Add strict source selection

def select_relevant_sources(self, question, results):
    """
    Select only retrieved verses that directly answer the question.
    """
    if not results:
        return []

    source_parts = []

    for result in results:
        source_parts.append(
            f"رتبه {result['rank']}\n"
            f"[سوره {result['surah']}، آیه {result['ayah']}]\n"
            f"ترجمه فولادوند: {result['fooladvand']}\n"
            f"ترجمه انصاریان: {result['ansarian']}"
        )

    selection_prompt = f"""
فقط آیه‌هایی را انتخاب کن که مستقیماً و صریحاً پاسخ سؤال را می‌دهند.

قوانین:
1. فقط شماره رتبه منابع مرتبط را برگردان.
2. هیچ پاسخ یا توضیحی تولید نکن.
3. اگر هیچ منبعی پاسخ مستقیم ندارد، آرایه خالی برگردان.
4. فقط JSON معتبر با قالب زیر برگردان:
{{"selected_ranks": [1, 2]}}

سؤال:
{question.strip()}

منابع:
{chr(10).join(source_parts)}
""".strip()

    response = self.client.chat.completions.create(
        model=self.chat_model,
        messages=[
            {
                "role": "user",
                "content": selection_prompt
            }
        ],
        temperature=0,
        max_tokens=100
    )

    raw_output = response.choices[0].message.content.strip()

    try:
        json_start = raw_output.find("{")
        json_end = raw_output.rfind("}") + 1

        if json_start == -1 or json_end <= json_start:
            return []

        parsed_output = json.loads(
            raw_output[json_start:json_end]
        )

        selected_ranks = parsed_output.get(
            "selected_ranks",
            []
        )

        valid_ranks = {
            result["rank"]
            for result in results
        }

        selected_ranks = [
            rank
            for rank in selected_ranks
            if isinstance(rank, int) and rank in valid_ranks
        ]

        return [
            result
            for result in results
            if result["rank"] in selected_ranks
        ]

    except (json.JSONDecodeError, TypeError, ValueError):
        return []


QuranRAGPipeline.select_relevant_sources = select_relevant_sources

print("Source selection method is ready.")

Source selection method is ready.


<div style="direction: rtl; white-space: normal; line-height: 1;">
در این بخش، پاسخ نهایی مستقیماً از ترجمه معتبر آیات انتخاب‌شده ساخته می‌شود و مدل اجازه بازنویسی آزاد ندارد
</div>

In [22]:
# Cell 4 - Build the verified answer from trusted Quran texts

def build_verified_answer(self, selected_sources):
    """
    Build the final answer directly from trusted Quran texts.
    """
    if not selected_sources:
        return (
            "در آیات بازیابی‌شده، "
            "پاسخ مستقیم و کافی پیدا نشد."
        )

    answer_parts = []

    for source in selected_sources:
        answer_parts.append(
            f"متن آیه:\n"
            f"{source['arabic']}\n\n"
            f"ترجمه فولادوند:\n"
            f"{source['fooladvand']}\n\n"
            f"ترجمه انصاریان:\n"
            f"{source['ansarian']}\n\n"
            f"[سوره {source['surah']}، آیه {source['ayah']}]"
        )

    return "\n\n---\n\n".join(answer_parts)


QuranRAGPipeline.build_verified_answer = build_verified_answer

print("Verified answer builder is ready.")

Verified answer builder is ready.


<div style="direction: rtl; white-space: normal; line-height: 1;">
در این بخش، کل مسیر بازیابی، انتخاب منابع معتبر، ساخت پاسخ نهایی و cache در یک متد ask اجرا می‌شود.
</div>

In [23]:
# Add the main verified RAG method

def ask(self, question, top_k=None):
    """
    Run the complete verified Quran RAG pipeline.
    """
    if not isinstance(question, str) or not question.strip():
        raise ValueError("Question must be a non-empty string.")

    selected_top_k = top_k or self.top_k

    original_top_k = self.top_k
    self.top_k = selected_top_k
    cache_key = self._make_cache_key(question)
    self.top_k = original_top_k

    if self.use_cache and cache_key in self.cache:
        cached_result = self.cache[cache_key].copy()
        cached_result["from_cache"] = True
        return cached_result

    retrieved_sources = self.retrieve(
        question=question,
        top_k=selected_top_k
    )

    selected_sources = self.select_relevant_sources(
        question=question,
        results=retrieved_sources
    )

    answer = self.build_verified_answer(
        selected_sources=selected_sources
    )

    final_result = {
        "question": question.strip(),
        "answer": answer,
        "sources": selected_sources,
        "retrieved_sources": retrieved_sources,
        "from_cache": False,
        "embedding_model": self.embedding_model_name,
        "generation_model": self.chat_model,
        "top_k": selected_top_k
    }

    if self.use_cache:
        self.cache[cache_key] = final_result
        self._save_cache()

    return final_result


QuranRAGPipeline.ask = ask

print("Main verified RAG method is ready.")

Main verified RAG method is ready.


<div style="direction: rtl; white-space: normal; line-height: 1;">
این سلول کل پایپ‌لاین را بدون cache آزمایش می‌کند و پاسخ، منابع انتخاب‌شده و همه آیات بازیابی‌شده را نمایش می‌دهد.
</div>

In [24]:
# Cell 6 - Test the complete verified Quran RAG pipeline

rag_test = QuranRAGPipeline(
    top_k=3,
    use_cache=False
)

result = rag_test.ask(
    question="خدا چه کسانی را دوست دارد؟"
)

print("Answer:")
print(result["answer"])

print("\nSelected sources:")
for source in result["sources"]:
    print(
        f"- Surah {source['surah']}, "
        f"Ayah {source['ayah']}, "
        f"Score: {source['score']:.4f}"
    )

print("\nAll retrieved sources:")
for source in result["retrieved_sources"]:
    print(
        f"- Surah {source['surah']}, "
        f"Ayah {source['ayah']}, "
        f"Score: {source['score']:.4f}"
    )

Answer:
متن آیه:
إِنَّ ٱللَّهَ يُحِبُّ ٱلَّذِينَ يُقَٰتِلُونَ فِى سَبِيلِهِۦ صَفًّا كَأَنَّهُم بُنْيَٰنٌ مَّرْصُوصٌ

ترجمه فولادوند:
در حقیقت، خدا دوست دارد کسانی را که در راه او صف در صف، چنانکه گویی بنایی ریخته شده از سرب‌اند، جهاد می‌کنند.

ترجمه انصاریان:
خدا کسانی را دوست دارد که صف زده در راه او جهاد می کنند [و از ثابت قدمی] گویی بنایی پولادین و استوارند.

[سوره 61، آیه 4]

Selected sources:
- Surah 61, Ayah 4, Score: 0.8680

All retrieved sources:
- Surah 61, Ayah 4, Score: 0.8680
- Surah 2, Ayah 165, Score: 0.8619
- Surah 42, Ayah 19, Score: 0.8610
